In [1]:
!pip3 install pyvisa-py pyserial pyusb zeroconf tqdm

  Obtaining dependency information for pyvisa-py from https://files.pythonhosted.org/packages/70/0b/06ccf917ce30d78d75e452d0afc89236b1d384b776c5c33d75c8cc55fbe7/pyvisa_py-0.8.1-py3-none-any.whl.metadata
  Using cached pyvisa_py-0.8.1-py3-none-any.whl.metadata (6.8 kB)
  Obtaining dependency information for pyserial from https://files.pythonhosted.org/packages/07/bc/587a445451b253b285629263eb51c2d8e9bcea4fc97826266d186f96f558/pyserial-3.5-py2.py3-none-any.whl.metadata
  Using cached pyserial-3.5-py2.py3-none-any.whl.metadata (1.6 kB)
  Obtaining dependency information for pyusb from https://files.pythonhosted.org/packages/28/b8/27e6312e86408a44fe16bd28ee12dd98608b39f7e7e57884a24e8f29b573/pyusb-1.3.1-py3-none-any.whl.metadata
  Using cached pyusb-1.3.1-py3-none-any.whl.metadata (2.5 kB)
  Obtaining dependency information for zeroconf from https://files.pythonhosted.org/packages/cd/6a/1e8a987a2b2805a59d355cd2e1d5fce03bc1da8859a0efcad4edb26892af/zeroconf-0.149.16-cp311-cp311-manylinux2014_

In [2]:
import pyvisa
import re
import csv
import numpy as np
import matplotlib.pyplot as plt
from time import sleep, perf_counter
from tqdm import tqdm

In [3]:
class GPIB:
    """Use GPIB devices via Prologix GPIB-USB Controller Rev 6.4.1"""
    def __init__(self, dev="ttyUSBGPIB", addr=None, res=None):
        self._rm   = pyvisa.ResourceManager("@py")               # using PyVISA-py
        self._dev  = self.setDev(dev)   if dev  else None
        self._addr = self.setAddr(addr) if addr else None
        self._res  = self.setRes(res)   if res  else self._res
        self._esc  = ['\x1b', '\x0d', '\x0a', '\x2b']

    def setDev(self, dev=None):
        if dev:
            self._dev = str(dev)
        if self._rm and self._dev:
            for res in sorted(self._rm.list_resources()):
                if self._dev in res:
                    self._res = self._rm.open_resource(res)
                    self.setRes(self._res)
        return self._dev
    
    def setRes(self, res=None):
        if res:
            self._res = res
        if self._res:
            self._res.read_termination  = "\n"
            self._res.write_termination = "\n"
            self._res.write("++mode 1") # "++mode" is "0" (GPIB TALKER or GPIB LISTENER) or "1" (Controller-In-Charge (CIC))
        return self._res        

    def setAddr(self, addr=None):
        if addr:
            self._addr = int(addr)
        if self._res and self._addr:
            self._res.write("++addr " + str(self._addr))
        return self._addr
                
    def _escape(self, data):
        # xlat_t = { ord(c):'\x1b' + c for c in '\x1b\r\n+' }; '+A\rB\nC\x1b'.translate(xlat_t) #==> '\x1b+A\x1b\rB\x1b\nC\x1b\x1b'
        for b in ['\x1b', '\x0d', '\x0a', '\x2b']: # ESC, CR, LF, PLUS (ESC MUST BE FIRST)
            data = data.replace(b, '\x1b' + b) # prepend with ESC
        return data
            
    def write(self, addr, data):
        if addr and data:
            self.setAddr(addr)
            if self._res:
                return self._res.write(self._escape(data))

    def query(self, addr, data, dtype="float"):
        if addr and data and dtype:
            self.setAddr(addr)
            if self._res:
                if dtype == "float":
                    return float(re.sub('[^0-9eE\.+-]', '',
                        self._res.query(self._escape(data))
                    ))
                else:
                    return self._res.query(data)

In [4]:
class Resource(GPIB):
    def set_freq(self, freq: float):
        """Set the frequency for the Synthesized Function Generator"""
        self.write(19, f"FREQ {freq:.2f}")

    def get_freq(self) -> float:
        """Get the frequency form the Synthesized Function Generator"""
        return self.query(19, f"FREQ?")
    
    def get_output(self):
        """Get the two output signal from the lock-in amplifier"""
        return (self.query(7, "QX"), self.query(7, "Qy"))

    def set_temp(self, U: float):
        """Set the temperature for the petier element"""  
        self.write(7, f"X5,{U:.3f}")

    def get_temp(self) -> float:
        """Get the temperatur from the thermometer"""
        return (self.query(7, "X1") - 1.000325) * 259.2684

    def reset_temp(self):
        """Set the temperatur for the petier element to zero"""
        self.write(7, "X5,0.000")

In [42]:
class DataWriter(Resource):
    def _create_data_file(self, filename:str , fieldnames):
        """Creates a file headers with given file name"""
        with open(f"{filename}.csv", "w") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
        
    def _write_data_entry(self, data, filename: str, fieldnames):
        with open(f"{filename}.csv", "a") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writerow(data)

    def create_file_for_xy(self, filename, nom):
        fieldnames = ["freq"] + [f"X{i}" for i in range(nom)] + [f"Y{i}" for i in range(nom)]
        self._create_data_file(filename, fieldnames)
    
    def write_xy(self, filename, freq, x, y, nom):
        fieldnames = ["freq"] + [f"X{i}" for i in range(nom)] + [f"Y{i}" for i in range(nom)]
        #print({**{"Freq": freq}, **dict(zip([f"X{i}"for i in range(1,21)], x)), **dict(zip([f"Y{i}" for i in range(1,21)], y))})
        self._write_data_entry({**{"freq": freq}, **dict(zip([f"X{i}"for i in range(nom)], x)), **dict(zip([f"Y{i}" for i in range(nom)], y))}, filename, fieldnames)        
    
    def write_ampl(self, freq, ampl, ampl_err, phase, phase_err, filename):
        fieldnames = ["freq", "amplitude", "amplitude error", "phase", "phase error"]
        self._create_data_file(filename, fieldnames)
        for i in range(len(freq)):
            self._write_data_entry({"freq": freq[i], "amplitude": ampl[i], "amplitude error": ampl_err[i], "phase":phase[i], "phase error":phase_err[i]}, filename, fieldnames)

    def create_file_for_temp(self, filename):
        fieldnames = ["time", "temperatur"]
        self._create_data_file(filename, fieldnames)
    
    def write_temp(self, time, temp, filename):
        fieldnames = ["time", "temperatur"]
        self._write_data_entry({"time": time, "temperatur": temp}, filename, fieldnames)

    def create_file_for_temp_xy(self, filename, nom):
        fieldnames = ["temperatur"] + ["freq"] + [f"X{i}" for i in range(nom)] + [f"Y{i}" for i in range(nom)]
        self._create_data_file(filename, fieldnames)

    def write_tem_xy(self, filename, temp, freq, x, y, nom):
        fieldnames = ["temperatur", "freq"] + [f"X{i}" for i in range(nom)] + [f"Y{i}" for i in range(nom)]
        self._write_data_entry({**{"temperatur": temp, "freq": freq}, **dict(zip([f"X{i}"for i in range(nom)], x)), **dict(zip([f"Y{i}" for i in range(nom)], y))}, filename, fieldnames)

    def write_tem_ampl(self, temp, freq, ampl, ampl_err, filename):
        fieldnames = ["temp", "freq", "amplitude", "amplitude error"]
        self._create_data_file(filename, fieldnames)
        for i in range(len(freq)):
            self._write_data_entry({"temp": temp, "freq": freq[i], "amplitude": ampl[i], "amplitude error": ampl_err[i]}, filename, fieldnames)

# Experiments

In [46]:
class ExperimentBase(DataWriter):
    def scan_band(self, start: float, end: float, filename, step=1, nom=20):
        self.create_file_for_xy(filename, nom)
        
        frequencies = np.arange(start, end, step)
        output_x = np.zeros((len(frequencies), nom))
        output_y = np.zeros((len(frequencies), nom))
        
        for i, f in tqdm(enumerate(frequencies), total=frequencies.shape[0]):
            self.set_freq(f)
            sleep(2)
            for measure in range(nom):
                sleep(1.5)
                output_x[i,measure], output_y[i,measure] = self.get_output()
            self.write_xy(filename, f, output_x[i], output_y[i], nom)
                
        return frequencies, output_x, output_y

    def scan_band_temp(self, start: float, end: float, filename, step=1, nom=20):
        self.create_file_for_temp_xy(filename, nom)
        
        frequencies = np.arange(start, end, step)
        output_x = np.zeros((len(frequencies), nom))
        output_y = np.zeros((len(frequencies), nom))
        
        for i, f in tqdm(enumerate(frequencies), total=frequencies.shape[0]):
            self.set_freq(f)
            sleep(0.5)
            temp = self.get_temp()
            for measure in range(nom):
                sleep(0.1)
                output_x[i,measure], output_y[i,measure] = self.get_output()
            self.write_tem_xy(filename, temp, f, output_x[i], output_y[i], nom)
                
        return frequencies, output_x, output_y

    def calc_ampl_and_phase(self, x, y):
        ampl = np.sqrt(x**2 + y**2)
        phase = np.arctan2(y,x)

        ampl_mean = np.mean(ampl, axis=1)
        phase_mean = np.mean(phase, axis=1)

        ampl_err = np.std(ampl, axis=1) / np.sqrt(ampl.shape[1])
        phase_err = np.std(phase, axis=1) /  np.sqrt(phase.shape[1])

        return ampl_mean, ampl_err, phase_mean, phase_err

    def plot_ampl(self, freq, ampl, ampl_err, filename=None):
        fig, ax = plt.subplots()
        ax.errorbar(freq, ampl, yerr=ampl_err, fmt=".", capsize=2)
        ax.set_xlabel("frequencies [Hz]")
        ax.set_ylabel("amplitude b.E.")
        ax.grid(alpha=0.5, ls="--")
        if filename:
            fig.savefig(filename)

    def plot_phase(self, freq, phase, phase_err, filename=None):
        fig, ax = plt.subplots()
        ax.errorbar(freq, phase, yerr=phase_err, fmt=".", capsize=2)
        ax.set_xlabel("frequencies [Hz]")
        ax.set_ylabel("phase")
        ax.grid(alpha=0.5, ls="--")
        if filename:
            fig.savefig(filename)

In [50]:
class Experiments(ExperimentBase):
    def experiment0(self):
        freq, x, y = self.scan_band(start=750, end=1000, step=1, nom=2, filename="full-band-raw")
        freq2 = freq * 2
        
        A_mean, A_err, ph_mean, ph_err = self.calc_ampl_and_phase(x,y)
        self.write_ampl(freq2, A_mean, A_err, ph_mean, ph_err, "full-band-phampl")
        self.plot_ampl(freq2, A_mean, A_err, "full-band_ampl.png")
        
    def experiment1(self):
        sleep(2)

        # first resonance frequency
        freq, x, y = self.scan_band(start=100.0, end=200.0, step=0.25, nom=20, filename="freq1-band-raw")
        freq2 = freq * 2
        
        A_mean, A_err, ph_mean, ph_err = self.calc_ampl_and_phase(x,y)
        self.write_ampl(freq2, A_mean, A_err, ph_mean, ph_err, "freq1-band-phampl")
        self.plot_ampl(freq2, A_mean, A_err, "freq1-band_ampl.png")
        self.plot_phase(freq2, ph_mean, ph_err, "freq1-band_phase.png")

        # second resonance frequency
        freq, x, y = self.scan_band(start=835, end=985, step=0.25, nom=20, filename="freq2-band-raw")
        freq2 = freq * 2
        
        #A_mean, A_err, ph_mean, ph_err = self.calc_ampl_and_phase(x,y)
        self.write_ampl(freq2, A_mean, A_err, ph_mean, ph_err, "freq2-band-phampl")
        self.plot_ampl(freq2, A_mean, A_err, "freq2-band_ampl.png")
        self.plot_phase(freq2, ph_mean, ph_err, "freq2-band_phase.png")

        # thired resonance frequency
        freq, x, y = self.scan_band(start=2450, end=2650, step=0.25, nom=20, filename="freq3-band-raw")
        freq2 = freq * 2
        
        #A_mean, A_err, ph_mean, ph_err = self.calc_ampl_and_phase(x,y)
        self.write_ampl(freq2, A_mean, A_err, ph_mean, ph_err, "freq3-band-phampl")
        self.plot_ampl(freq2, A_mean, A_err, "freq3-band_ampl.png")
        self.plot_phase(freq2, ph_mean, ph_err, "freq3-band_phase.png")

    def experiment2(self):
        try:
            wait_time = 30*60
            wait_start = 60*60
            
            self.reset_temp()
            sleep(wait_start)
    
            for i in np.arange(0.0,0.7,0.1):
                self.set_temp(i)
                sleep(wait_time)
    
                freq, x, y = self.scan_band_temp(start=100.0, end=190.0, step=1, nom=20, filename=f"temp{i}_freq-band-raw")
                freq2 = freq * 2
    
                temp = self.get_temp()
                A_mean, A_err, ph_mean, ph_err = self.calc_ampl_and_phase(x,y)
                self.write_tem_ampl(temp, freq2, A_mean, A_err, f"temp{i}_freq-band-phampl")
                self.plot_ampl(freq2, A_mean, A_err, f"temp{i}_freq-band_ampl.png")
        finally:
            self.reset_temp()
            
            

exp = Experiments()
# exp.experiment1()
exp.experiment0()

  0%|          | 1/250 [00:07<31:35,  7.61s/it]


KeyboardInterrupt: 

In [ ]:
alpha0 = 1.424987
d = 200e-6
l=2e-2
E=98e9
rho=8.5e3

alpha0 * np.pi/(16*np.sqrt(3))*d/l**2*np.sqrt(E/rho)